# 04 - Chunking and Splitting

In [1]:
!pip install -qU langchain_community
!pip install -qU langchain_experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 

In [2]:
import json
import re
import hashlib
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 1. Konfigurasi

In [3]:
PIPELINE_VERSION = "chunking_v1"
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 200
 
STRUCTURED_DIR = Path("/kaggle/input/notebooks/muhammadnabilul/03-structured-cleaned-extract/structured")
OUTPUT_CHUNKS_DIR = Path("/kaggle/working/chunks")
OUTPUT_CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
 
QA_CANDIDATES = {
    "mixed_rich":  ["statistical", "comparative", "interpretative"],
    "table_rich":  ["statistical", "comparative"],
    "figure_rich": ["interpretative", "comparative", "trend"],
    "text_only":   ["definition", "factual", "summary"]
}
 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""]
)

## 2. Filter Noise

In [4]:
NOISE_TITLE_PATTERNS = [
    r'^\s*[a-zA-Z]\.\s*$',
    r'^\s*\d+\.\s*$',
    r'daftar\s+(isi|gambar|tabel|lampiran)',
    r'kata\s+pengantar',
    r'tim\s+penyusun',
    r'^pengarah',
    r'^penyunting',
    r'^penanggung\s+jawab',
    r'^\s*\*{0,4}\s*$',
]
 
NOISE_CONTENT_PATTERNS = [
    r'\.{5,}',
    r'^[xXiIvV]{1,5}$',
]

In [5]:
def is_noise_section(section: dict) -> bool:
    raw_title = re.sub(r'[*_`#]', '', section.get("title", "")).strip().lower()
 
    for pat in NOISE_TITLE_PATTERNS:
        if re.search(pat, raw_title, re.IGNORECASE):
            return True
 
    blocks = section.get("blocks", [])
    all_text = "\n".join(b.get("text", "") for b in blocks)
    clean_text = re.sub(r'[#*_`|]', '', all_text).strip()
 
    lines = [l.strip() for l in clean_text.splitlines() if l.strip()]
    if not lines:
        return True
 
    noise_count = sum(
        1 for l in lines
        if any(re.search(p, l) for p in NOISE_CONTENT_PATTERNS)
    )
    
    if noise_count / len(lines) > 0.5:
        return True
 
    return False

## 3. Fungsi-Fungsi

In [6]:
def clean_block_text(text):
    # heading markdown
    text = re.sub(r'^#{1,6}\s*', '', text, flags=re.MULTILINE)

    # bold
    text = re.sub(r'\*{1,3}([^*\n]+)\*{1,3}', r'\1', text)

    # italic
    text = re.sub(r'_{1,2}([^_\n]+)_{1,2}', r'\1', text)

    # inline code
    text = re.sub(r'`([^`]+)`', r'\1', text)

    return text.strip()

In [7]:
def build_master_chunks(doc: dict) -> list[dict]:
    metadata = doc.get("document_metadata", {})
    base_name = metadata.get("nama_file", "doc").replace(".pdf", "")
    all_chunks = []
 
    for idx, section in enumerate(doc.get("sections", [])):
        if is_noise_section(section):
            continue
 
        blocks = section.get("blocks", [])
        if not blocks:
            continue
 
        seen_ids = set()
        unique_blocks = []
        for b in blocks:
            cid = b.get("chunk_id", "")
            if cid not in seen_ids:
                seen_ids.add(cid)
                unique_blocks.append(b)
 
        section_text = "\n\n".join(
            clean_block_text(b["text"])
            for b in unique_blocks
            if b.get("text", "").strip()
        )
 
        if not section_text.strip():
            continue
 
        raw_sid = str(section.get("section_id", f"sec_{idx:03d}"))
        section_id = re.sub(r'[^a-zA-Z0-9_]', '_', raw_sid)
        section_title = re.sub(r'[*_`#]', '', section.get("title", "No Title")).strip()
 
        source_pages = sorted(set(b["page_number"] for b in unique_blocks))
        source_chunk_ids = sorted(set(b["chunk_id"] for b in unique_blocks))
 
        sub_texts = text_splitter.split_text(section_text)
        total_sub = len(sub_texts)
 
        for i, text in enumerate(sub_texts):
            tl = text.lower()
            has_table = (
                bool(re.search(r'\btabel\s+\d+', tl)) or
                (text.count('|') >= 4 and '\n' in text)
            )
            has_figure = bool(re.search(r'\b(gambar|grafik|diagram)\s+\d+', tl))
 
            if has_table and has_figure:
                chunk_type = "mixed_rich"
            elif has_table:
                chunk_type = "table_rich"
            elif has_figure:
                chunk_type = "figure_rich"
            else:
                chunk_type = "text_only"
 
            chunk = {
                "chunk_id":                 f"{base_name}__{section_id}__c{i:03d}",
                "context_hash":             hashlib.md5(text.encode()).hexdigest(),
                "pipeline_version":         PIPELINE_VERSION,
 
                "document_title":           metadata.get("judul"),
                "document_year":            metadata.get("tahun"),
                "source_file":              metadata.get("nama_file"),
                "bps_url":                  metadata.get("bps_url"),
 
                "section_id":               section_id,
                "section_title":            section_title,
                "section_level":            section.get("level", 0),
                "section_source":           section.get("source", "unknown"),
 
                "page_start":               section.get("page_start"),
                "page_end":                 section.get("page_end"),
                "source_pages":             source_pages,
                "source_chunk_ids":         source_chunk_ids,
 
                "chunk_index":              i,
                "total_chunks_in_section":  total_sub,
                "is_first_chunk":           (i == 0),
                "is_last_chunk":            (i == total_sub - 1),
 
                "table_count":              len(section.get("tables", [])),
                "figure_count":             len(section.get("figures", [])),
 
                "chunk_type":               chunk_type,
                "qa_candidate":             QA_CANDIDATES[chunk_type],
                "estimated_qa_count":       len(QA_CANDIDATES[chunk_type]),
 
                "text":                     text,
                "char_count":               len(text),
                "word_count":               len(text.split()),
            }
            
            all_chunks.append(chunk)
 
    return all_chunks

## 4. Main Program

In [8]:
structured_files = sorted(STRUCTURED_DIR.glob("*.json"))
print(f"Menjalankan {PIPELINE_VERSION} untuk {len(structured_files)} dokumen BPS...\n")
 
total_chunks = 0
for file_path in structured_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        doc = json.load(f)
 
    chunks = build_master_chunks(doc)
    total_chunks += len(chunks)
 
    out_path = OUTPUT_CHUNKS_DIR / f"{file_path.stem}_chunks.json"
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
 
    n_sections = len(doc.get("sections", []))
    n_skipped = sum(1 for s in doc["sections"] if is_noise_section(s))
    print(f"  {file_path.name}")
    print(f"    Sections: {n_sections} total | {n_sections - n_skipped} diproses | {n_skipped} di-skip")
    print(f"    Chunks  : {len(chunks)}")
 
print(f"\nSELESAI. Total: {total_chunks} Master Chunks siap untuk FAISS & QA Generator.")

Menjalankan chunking_v1 untuk 47 dokumen BPS...

  analisis-petty-corruption-hasil-spak-2020---2024.json
    Sections: 115 total | 105 diproses | 10 di-skip
    Chunks  : 395
  indeks-pembangunan-manusia-2022.json
    Sections: 230 total | 224 diproses | 6 di-skip
    Chunks  : 450
  indeks-pembangunan-manusia-2023.json
    Sections: 125 total | 119 diproses | 6 di-skip
    Chunks  : 317
  indeks-pembangunan-manusia-2024.json
    Sections: 186 total | 178 diproses | 8 di-skip
    Chunks  : 370
  indeks-perilaku-anti-korupsi-2023.json
    Sections: 94 total | 87 diproses | 7 di-skip
    Chunks  : 195
  indeks-perilaku-anti-korupsi-2024.json
    Sections: 117 total | 109 diproses | 8 di-skip
    Chunks  : 252
  indikator-kesejahteraan-rakyat-2023.json
    Sections: 226 total | 220 diproses | 6 di-skip
    Chunks  : 607
  indikator-kesejahteraan-rakyat-2024.json
    Sections: 175 total | 168 diproses | 7 di-skip
    Chunks  : 606
  indikator-kesejahteraan-rakyat-2025.json
    Sections: 13